# Upload zipped directory and unzip it in Azure blob storage

In [ ]:
import os
import zipfile
from azure.storage.blob import BlobServiceClient

In [ ]:
# Configuration
connect_str = ""  # Replace with your Azure Storage connection string
container_name = ""  # Replace with your container name
blob_prefix = "datasets/segmentation_data_zip/"  # Path to the zip files in blob storage
local_download_path = "./temp"  # Local directory for temporary storage
local_extract_path = "./temp/unzipped"  # Local directory for extracted files

In [ ]:
# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connect_str)
container_client = blob_service_client.get_container_client(container_name)

In [ ]:
# Ensure local directories exist
os.makedirs(local_download_path, exist_ok=True)
os.makedirs(local_extract_path, exist_ok=True)

In [ ]:
# List blobs in the target directory
blob_list = container_client.list_blobs(name_starts_with=blob_prefix)

In [ ]:
for blob in blob_list:
    # Download each zip file
    blob_client = container_client.get_blob_client(blob)
    local_zip_path = os.path.join(local_download_path, os.path.basename(blob.name))
    print(f"Downloading {blob.name} to {local_zip_path}")
    with open(local_zip_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())

    # Unzip the file
    print(f"Unzipping {local_zip_path}")
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_extract_path)

    # Upload extracted files back to the blob storage
    for root, _, files in os.walk(local_extract_path):
        for file in files:
            local_file_path = os.path.join(root, file)
            blob_path = os.path.join(blob_prefix, "unzipped", os.path.relpath(local_file_path, local_extract_path))
            print(f"Uploading {local_file_path} to {blob_path}")
            with open(local_file_path, "rb") as data:
                container_client.upload_blob(name=blob_path, data=data, overwrite=True)

    # Clean up local files
    os.remove(local_zip_path)
    for root, _, files in os.walk(local_extract_path):
        for file in files:
            os.remove(os.path.join(root, file))

print("All files unzipped and uploaded successfully.")

# Register Datasets

In [ ]:
import os
from azureml.core import Dataset, Workspace
import numpy as np

In [ ]:
ws = Workspace.from_config()

In [ ]:
datastore = ws.get_default_datastore()

In [ ]:
# Register the images dataset
images_train_dataset = Dataset.File.from_files(path=(datastore,'datasets/segmentation_data_zip/unzipped/leftImg8bit/train'))
images_train_dataset.register(workspace=ws,
                                name='segmentation_images_train',
                                description='[Training] Dataset containing input images for segmentation',
                                create_new_version=True)

In [ ]:
# Register the images dataset
images_train_dataset = Dataset.File.from_files(path=(datastore,'datasets/segmentation_data_zip/unzipped/leftImg8bit/val'))
images_train_dataset.register(workspace=ws,
                                name='segmentation_images_val',
                                description='[Validation] Dataset containing input images for segmentation',
                                create_new_version=True)

In [ ]:
# Register the images dataset
images_train_dataset = Dataset.File.from_files(path=(datastore,'datasets/segmentation_data_zip/unzipped/leftImg8bit/test'))
images_train_dataset.register(workspace=ws,
                                name='segmentation_images_test',
                                description='[Testing] Dataset containing input images for testing',
                                create_new_version=True)

In [ ]:
# Register the masks dataset
masks_dataset = Dataset.File.from_files(path=(datastore, 'datasets/segmentation_data_zip/unzipped/gtFine/train'))
masks_dataset.register(workspace=ws,
                       name='segmentation_masks_train',
                       description='[Train] Dataset containing masks for segmentation',
                       create_new_version=True)
# print("Datasets registered successfully!")

In [ ]:
# Register the masks dataset
masks_dataset = Dataset.File.from_files(path=(datastore, 'datasets/segmentation_data_zip/unzipped/gtFine/val'))
masks_dataset.register(workspace=ws,
                       name='segmentation_masks_val',
                       description='[Validation] Dataset containing masks for segmentation',
                       create_new_version=True)
# print("Datasets registered successfully!")

In [ ]:
# Register the masks dataset
masks_dataset = Dataset.File.from_files(path=(datastore, 'datasets/segmentation_data_zip/unzipped/gtFine/test'))
masks_dataset.register(workspace=ws,
                       name='segmentation_masks_test',
                       description='[Testing] Dataset containing masks for segmentation',
                       create_new_version=True)

# Load Dataset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from azureml.core import Dataset, Workspace
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from data_processing.image_processing import load_dataset_unet
from tqdm import tqdm

In [ ]:
ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep='\n')

In [ ]:
dataset_train, dataset_val, mount_pts = load_dataset_unet(batch_size=6)

In [ ]:
for name in mount_pts:
    print(mount_pts[name].mount_point)

In [ ]:
pip freeze > requirements_data_processing.txt

# Training Prep (Metrics, Cost Function, Optimization Algorithm, Callbacks \[early stopping, plotting, learning rate decay, Saving best five models / checkpoints, custom history class, tensorboard, mlflow])

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from callbacks.callbacks import keras_callbacks, PlotResultsCallback

from custom_metrics.custom_metrics import iou, dice_coefficient
from model_definitions.unet import unet_with_vgg16_encoder

from utils.utils import keras_optimizer

In [ ]:
# Define my Azure details
# subscription_id = "your_subscription_id"
# resource_group_name = "your_resource_group_name"
# workspace_name = "your_workspace_name"

# Get the Azure MLClient instance
# ml_client, ws = get_ml_client(ws.subscription_id, ws.resource_group, ws.name)

# Log or interact with Azure ML
# ml_client.jobs.upload(path="outputs/plots", name="epoch_plots")

In [ ]:
# from utils.azure_ml_utils import get_ml_client
# import keras
import keras
keras.backend.clear_session()

In [ ]:
early_stop, model_checkpoint, reduce_lr, custom_history, top_k_checkpoint = keras_callbacks()

In [ ]:
plot_callback=PlotResultsCallback(validation_data=dataset_val)

In [ ]:
# Combine custom and builtin metrics for Keras model.fit method

callbacks = [early_stop, model_checkpoint, reduce_lr, plot_callback, custom_history, top_k_checkpoint]
metrics = [dice_coefficient, iou, 'accuracy']

In [ ]:
input_shape = (1024, 2048, 3)  # Input shape for Cityscapes dataset
num_classes = 8  # Number of major groups/classes in mask
model = unet_with_vgg16_encoder(input_shape, num_classes)

In [ ]:
optimizer = keras_optimizer()
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=metrics)

In [ ]:
model.summary()

# Training

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [ ]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

In [ ]:
import mlflow
# import mlflow.azureml
from azureml.core import Workspace, Experiment
import mlflow.keras

In [ ]:
# Connect to your Azure ML workspace
ws = Workspace.from_config()

# Set the MLflow tracking URI to point to your Azure ML workspace
mlflow.set_tracking_uri(ws.get_mlflow_tracking_uri())

# Create an experiment in Azure ML
experiment_name = "unet_experiment_v2"
mlflow.set_experiment(experiment_name)

In [ ]:
# keras.config.disable_traceback_filtering()

In [ ]:
import os
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

In [ ]:
# mlflow.keras.autolog(disable=True)
# Enable autologging


In [ ]:
keras.__version__

In [ ]:
# batch size 6 with environment set to 1

In [ ]:
with mlflow.start_run():
    mlflow.keras.autolog()
    
    history = model.fit(
                    dataset_train.take(1),
                    validation_data=dataset_val(1),
                    epochs=10,
                    callbacks=callbacks)

In [ ]:
mlflow.end_run()

In [ ]:
history = custom_history.history

In [ ]:
# Convert NumPy arrays to lists for JSON serialization
import numpy as np

def convert_history(history):
    converted_history = {}
    for key, values in history.items():
        converted_values = []
        for value in values:
            if isinstance(value, (np.ndarray, np.number)):
                converted_values.append(value.tolist())
            else:
                converted_values.append(value)
        converted_history[key] = converted_values
    return converted_history

converted_history = convert_history(history)


In [ ]:
import json

with open('./outputs/training_history.json', 'w') as f:
    json.dump(converted_history, f)

In [ ]:
for images, masks in dataset_train.take(1):
    print("Images shape:", images.shape)
    print("Masks shape:", masks.shape)

# Deployment

# Monitoring